# Hands-on #06c — Snowpark with Python
**SnowPro Advanced: Architect** · Copyright © 2026 Innovation In Software Corporation. All rights reserved.

Smith Insurance's partner agencies send claims to S3 (Hands-on #06b). In this notebook you work with those claims **from Python in Amazon SageMaker Studio** — but the data never leaves Snowflake: Snowpark DataFrames are *lazy* and are translated to SQL that runs on the `SMITH_WH` warehouse.

| Part | What you do |
|---|---|
| 1 | Connect as the key-pair service user `SMITH_SNOWPARK_SVC` |
| 2 | Build lazy DataFrames and look at the SQL they generate |
| 3 | Flatten the JSON line items and aggregate per claim |
| 4 | Join to `CORE.POLICIES` and write `SNOWPARK.AGENCY_CLAIMS_CURATED` |
| 5 | Register a permanent Python **UDF** |
| 6 | Register a permanent Python **stored procedure** |
| 7 | Bring a small result back as pandas |
| A–C | Exercises — then return to `Day2/06c-Snowpark-Python.sql`, Part 2 |

**Prerequisites:** `bash data/aws/sagemaker-setup.sh` has run in a SageMaker Studio JupyterLab terminal (Snowpark + key pair), Hands-on #06c Demo 1 has created the user, and this notebook uses the **Python 3 (ipykernel)** kernel.

## Part 1 — Connect
Replace `<orgname>-<account_name>` with the value from Demo 1 (`SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME()`). The private key never leaves SageMaker Studio; Snowflake checks it against the public key on the user.

In [ ]:
import os
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType, FloatType, IntegerType

CONNECTION = {
    "account":          "<orgname>-<account_name>",
    "user":             "SMITH_SNOWPARK_SVC",
    "authenticator":    "SNOWFLAKE_JWT",
    "private_key_file": os.path.expanduser("~/.snowflake/rsa_key.p8"),
    "role":             "SMITH_ENGINEER",
    "warehouse":        "SMITH_WH",
    "database":         "SMITH_INS",
    "schema":           "SNOWPARK",
}

session = Session.builder.configs(CONNECTION).create()
session.sql("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_VERSION()").show()

## Part 2 — DataFrames are lazy
Nothing runs until an *action* (`show`, `collect`, `count`, `to_pandas`, `save_as_table`). Until then a DataFrame is a query plan.

In [ ]:
raw = session.table("SMITH_INS.RAW.AGENCY_CLAIMS_RAW")

claims = raw.select(
    F.col("RECORD")["claim_ref"].cast(StringType()).alias("CLAIM_REF"),
    F.col("RECORD")["agency"]["agency_id"].cast(StringType()).alias("AGENCY_ID"),
    F.col("RECORD")["agency"]["state"].cast(StringType()).alias("STATE"),
    F.col("RECORD")["line_of_business"].cast(StringType()).alias("LINE_OF_BUSINESS"),
).filter(F.col("LINE_OF_BUSINESS") == "AUTO")

print(type(claims))                      # still just a plan
print(claims.queries["queries"][-1])     # the SQL Snowpark will send

In [ ]:
claims.show(5)                           # action: the SQL runs on SMITH_WH
print("rows:", claims.count())

## Part 3 — Flatten the line items, aggregate per claim
`join_table_function("flatten", ...)` is `LATERAL FLATTEN`: one output row per array element, with the element in `VALUE`.

In [ ]:
def build_curated(session):
    raw = session.table("SMITH_INS.RAW.AGENCY_CLAIMS_RAW")
    lines = raw.join_table_function("flatten", F.col("RECORD")["line_items"]).select(
        F.col("RECORD")["claim_ref"].cast("string").alias("CLAIM_REF"),
        F.col("RECORD")["agency"]["agency_id"].cast("string").alias("AGENCY_ID"),
        F.col("RECORD")["agency"]["state"].cast("string").alias("STATE"),
        F.col("RECORD")["policy_id"].cast("string").alias("POLICY_ID"),
        F.col("RECORD")["line_of_business"].cast("string").alias("LINE_OF_BUSINESS"),
        F.col("RECORD")["loss_date"].cast("date").alias("LOSS_DATE"),
        F.col("RECORD")["reported_at"].cast("timestamp_ntz").alias("REPORTED_AT"),
        F.col("VALUE")["amount"].cast("number(12,2)").alias("AMOUNT"),
    )
    per_claim = lines.group_by("CLAIM_REF", "AGENCY_ID", "STATE", "POLICY_ID",
                               "LINE_OF_BUSINESS", "LOSS_DATE", "REPORTED_AT").agg(
        F.sum("AMOUNT").alias("TOTAL_AMOUNT"),
        F.count("AMOUNT").alias("LINE_ITEMS"),
    )
    policies = session.table("SMITH_INS.CORE.POLICIES").select(
        "POLICY_ID", F.col("STATUS").alias("POLICY_STATUS"), "ANNUAL_PREMIUM")
    return per_claim.join(policies, on="POLICY_ID", how="left")

curated = build_curated(session)
curated.show(5)

## Part 4 — Write the result back to Snowflake
`save_as_table` becomes one `CREATE TABLE ... AS SELECT`: the rows never travel to SageMaker Studio.

In [ ]:
curated.write.mode("overwrite").save_as_table("SMITH_INS.SNOWPARK.AGENCY_CLAIMS_CURATED")
session.table("SMITH_INS.SNOWPARK.AGENCY_CLAIMS_CURATED").count()

## Part 5 — A permanent Python UDF
The function is uploaded to `@SNOWPARK.CODE` and runs inside Snowflake. Once registered it is callable from SQL too (lab Exercise 2).

In [ ]:
@F.udf(name="SMITH_INS.SNOWPARK.SEVERITY_BAND", is_permanent=True,
       stage_location="@SMITH_INS.SNOWPARK.CODE", replace=True,
       input_types=[FloatType()], return_type=StringType())
def severity_band(amount):
    if amount is None:
        return None
    if amount < 2500:
        return "LOW"
    if amount < 10000:
        return "MEDIUM"
    return "HIGH"

(session.table("SMITH_INS.SNOWPARK.AGENCY_CLAIMS_CURATED")
    .with_column("SEVERITY", severity_band(F.col("TOTAL_AMOUNT")))
    .group_by("SEVERITY").count()
    .sort("SEVERITY")
    .show())

## Part 6 — A permanent stored procedure
Package the Part 3–4 logic so anyone (or a task) can rerun it with `CALL`.

In [ ]:
def refresh_agency_curated(session: Session) -> int:
    build_curated(session).write.mode("overwrite").save_as_table("SMITH_INS.SNOWPARK.AGENCY_CLAIMS_CURATED")
    return session.table("SMITH_INS.SNOWPARK.AGENCY_CLAIMS_CURATED").count()

session.sproc.register(
    refresh_agency_curated,
    name="SMITH_INS.SNOWPARK.REFRESH_AGENCY_CURATED",
    is_permanent=True, stage_location="@SMITH_INS.SNOWPARK.CODE", replace=True,
    return_type=IntegerType(), input_types=[],
    packages=["snowflake-snowpark-python"],
)
session.call("SMITH_INS.SNOWPARK.REFRESH_AGENCY_CURATED")

## Part 7 — Small results to pandas
Aggregate in Snowflake first; pull only the answer.

In [ ]:
by_lob = (session.table("SMITH_INS.SNOWPARK.AGENCY_CLAIMS_CURATED")
          .group_by("LINE_OF_BUSINESS")
          .agg(F.count("CLAIM_REF").alias("CLAIMS"), F.avg("TOTAL_AMOUNT").alias("AVG_AMOUNT"))
          .sort(F.col("CLAIMS").desc())
          .to_pandas())
by_lob

---
## Exercise A — Top agencies
Using only the DataFrame API on `SNOWPARK.AGENCY_CLAIMS_CURATED`, show the **top 5 agencies** (AGENCY_ID, STATE) by number of claims, with their total claimed amount.

In [ ]:
# YOUR CODE HERE

## Exercise B — Late reports
A claim is *late* when it was reported more than **14 days** after the loss date. Add a `DAYS_TO_REPORT` column (`F.datediff`), keep the late claims, and save them as **`SMITH_INS.SNOWPARK.LATE_REPORTS`**. How many are there? (You will find this query in Snowflake in lab Exercise 3.)

In [ ]:
# YOUR CODE HERE

## Exercise C — pandas → Snowflake
Underwriting keeps monthly claim **targets** per agency in a spreadsheet. Build this pandas DataFrame, write it to `SMITH_INS.SNOWPARK.AGENCY_TARGETS` with `session.write_pandas(..., auto_create_table=True, overwrite=True)`, then join it to the curated table to show each agency's claims vs. target.

```python
import pandas as pd
targets = pd.DataFrame({"AGENCY_ID": ["AG-101","AG-102","AG-103","AG-104","AG-105","AG-106","AG-107","AG-108"],
                        "MONTHLY_TARGET": [1500, 1500, 1400, 1300, 1300, 1200, 1200, 1400]})
```

In [ ]:
# YOUR CODE HERE

---
## Finish
Close the session, then return to **`Day2/06c-Snowpark-Python.sql` — Part 2** to check in Snowflake what this notebook did.

In [ ]:
session.close()